# Bài 16: Conversation Memory + Streamlit UI
## Hệ thống nhớ lịch sử hội thoại và giao diện web

**Mục tiêu:**
- Hiểu `MemorySaver` — LangGraph lưu state giữa các lần gọi
- Hiểu `st.session_state` — Streamlit lưu dữ liệu giữa các rerun
- Build Streamlit chat app tích hợp hệ thống phân tích cổ phiếu

---
## Phần 1: Lý thuyết

### Vấn đề: Hệ thống hiện tại không nhớ gì

Mỗi lần gọi `graph.invoke()` là một phiên hoàn toàn mới:
```
User: "Doanh thu NVDA bao nhiêu?"
Bot:  "$44 tỷ"

User: "Còn AMD thì sao?"   ← Bot không biết đang so sánh gì
Bot:  "Xin lỗi, bạn hỏi về gì?"
```

---

### Giải pháp 1: LangGraph MemorySaver

`MemorySaver` là một **checkpointer** — lưu state của graph sau mỗi lần chạy.
Cùng `thread_id` → graph đọc lại state cũ trước khi chạy tiếp.

```python
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)  # gắn vào graph

config = {"configurable": {"thread_id": "session_abc"}}

# Turn 1
graph.invoke({"messages": [{"role": "user", "content": "..."}]}, config=config)
# Turn 2 — cùng thread_id → graph nhớ Turn 1
graph.invoke({"messages": [{"role": "user", "content": "..."}]}, config=config)
```

**Trick quan trọng**: State cần dùng `Annotated[list, operator.add]` thay vì `list` bình thường —
khi đó LangGraph **cộng dồn** (append) vào list thay vì ghi đè.

---

### Giải pháp 2: Streamlit `st.session_state`

Streamlit **rerun toàn bộ script** mỗi khi user tương tác. `st.session_state` giữ dữ liệu giữa các rerun:

```python
if "messages" not in st.session_state:
    st.session_state.messages = []   # khởi tạo 1 lần duy nhất

st.session_state.messages.append({"role": "user", "content": question})
```

**Trong bài này**: `st.session_state` quản lý lịch sử hiển thị trên UI,
còn `MemorySaver` quản lý state của LangGraph graph.

---
## Phần 2: Ví dụ — MemorySaver với chatbot đơn giản

Demo: chatbot nhớ lịch sử hội thoại qua nhiều turn.

In [2]:
import os
from operator import add
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from google import genai
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [3]:
# ĐÃ CHO SẴN — chatbot với MemorySaver, đọc kỹ trước khi làm bài tập

class SimpleChatState(TypedDict):
    # Annotated[list, add] → mỗi turn CỘNG DỒN vào list thay vì ghi đè
    messages: Annotated[list[dict], add]


def chatbot_node(state: SimpleChatState):
    # Build lịch sử hội thoại thành text cho LLM
    history = ""
    for msg in state["messages"]:
        role = "Người dùng" if msg["role"] == "user" else "Assistant"
        history += f"{role}: {msg['content']}\n"

    prompt = (
        f"Đây là lịch sử hội thoại:\n{history}\n"
        "Hãy trả lời tin nhắn cuối của người dùng. "
        "Bạn là trợ lý phân tích cổ phiếu."
    )
    response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    return {"messages": [{"role": "assistant", "content": response.text}]}


# Build graph với MemorySaver
memory = MemorySaver()
builder = StateGraph(SimpleChatState)
builder.add_node("chat", chatbot_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)
chat_graph = builder.compile(checkpointer=memory)

print("Graph với MemorySaver đã sẵn sàng")

Graph với MemorySaver đã sẵn sàng


In [4]:
# Test 3 turns — cùng thread_id để graph nhớ lịch sử
config = {"configurable": {"thread_id": "demo_1"}}

turns = [
    "Doanh thu NVDA quý gần nhất là bao nhiêu?",
    "Còn AMD thì sao?",                          # không nhắc NVDA — bot phải nhớ context
    "So sánh hai công ty đó cho tôi",            # "hai công ty đó" — bot phải nhớ cả 2
]

for i, question in enumerate(turns, 1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {question}")
    result = chat_graph.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config
    )
    # Lấy tin nhắn cuối cùng (assistant vừa trả lời)
    last_msg = result["messages"][-1]
    print(f"Bot: {last_msg['content'][:300]}...")


--- Turn 1 ---
User: Doanh thu NVDA quý gần nhất là bao nhiêu?
Bot: Doanh thu của NVIDIA (NVDA) cho quý gần nhất là **26.04 tỷ USD**.

Đây là kết quả cho quý 1 năm tài chính 2025 của họ, kết thúc vào ngày 28 tháng 4 năm 2024. Con số này đã vượt qua đáng kể kỳ vọng của thị trường và cho thấy sự tăng trưởng mạnh mẽ, chủ yếu nhờ vào mảng trung tâm dữ liệu (AI)....

--- Turn 2 ---
User: Còn AMD thì sao?
Bot: Doanh thu của AMD cho quý gần nhất (quý 1 năm 2024, kết thúc vào ngày 30 tháng 3 năm 2024) là **5.47 tỷ USD**.

Con số này tăng nhẹ so với cùng kỳ năm trước, chủ yếu được thúc đẩy bởi sự tăng trưởng mạnh mẽ trong mảng trung tâm dữ liệu, đặc biệt là với dòng chip MI300 AI....

--- Turn 3 ---
User: So sánh hai công ty đó cho tôi
Bot: Được thôi, hãy cùng so sánh NVIDIA (NVDA) và AMD (Advanced Micro Devices), hai đối thủ chính trong ngành công nghiệp bán dẫn, dựa trên thông tin doanh thu gần đây và vị thế thị trường của họ.

---

**So sánh NVIDIA (NVDA) và AMD (Advanced Micro Devices)**



**Chú ý:** Turn 2 hỏi "Còn AMD thì sao?" mà không nhắc NVDA — bot trả lời đúng vì `MemorySaver` + `Annotated[list, add]` giữ toàn bộ messages từ Turn 1.

Thử đổi `thread_id` sang `"demo_2"` → bot không còn nhớ Turn 1 nữa.

---
## Phần 3: Bài tập — Xây dựng Streamlit App

File `phase4/app.py` đã được tạo sẵn với skeleton. Nhiệm vụ của bạn:

- ✅ Skeleton app, session state, chat UI — đã cho sẵn
- **❓ Bước 1:** Copy `collect_for_symbol()` từ Bài 15 vào app.py
- **❓ Bước 2:** Hoàn thiện `get_graph()` — build graph với MemorySaver
- **❓ Bước 3:** Điền phần gọi `graph.invoke()` trong chat handler

Chạy app:
```bash
streamlit run phase4/app.py
```